In [ ]:
import numpy as np

# Observations (two features per observation, normalized values for height and weight using z-score)
O = np.array([
    [1.2, 3.4],
    [2.1, 1.5],
    [0.8, 2.7],
    [3.2, 4.1],
    [1.5, 2.2],
    [4.1, 0.9],
    [2.7, 3.6],
    [0.5, 1.8],
    [3.8, 2.4],
    [1.9, 3.1]
])

# Boolean labels
y = np.array([
    True,
    False,
    True,
    False,
    True,
    False,
    True,
    False,
    True,
    False
]).astype(float)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
def evaluate(y_true, y_pred):
    tp = np.sum((y_true == 1) & (y_pred == 1))
    fp = np.sum((y_true == 0) & (y_pred == 1))
    fn = np.sum((y_true == 1) & (y_pred == 0))
    tn = np.sum((y_true == 0) & (y_pred == 0))

    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    f1 = (2 * precision * recall / (precision + recall) if precision + recall else 0.0)
    return precision, recall, f1

def sigmoid(z): #logistic regression
    z = np.clip(z, -500, 500)
    return 1.0 / (1.0 + np.exp(-z))

In [ ]:
# Hossein's Proposed Error (loss) Function: E = (1 - y_pred) for positive + y_pred for negative
# if y_batch = 1 -> (1-2*1) = -1 but if y_batch = 0 -> (1-2*0) = 1

def loss_function_hossein(y_batch, y_pred):
  return (1 - 2 * y_batch) * y_pred * (1 - y_pred) # Positive: -y_pred * (1-y_pred), Negative: +y_pred * (1-y_pred)

In [ ]:
# Standard binary cross-entropy
# Positive: -log(y_pred), Negative: -log(1-y_pred)
def loss_function_standard(y_batch, y_pred):
    #eps = 1e-15
    #y_pred = np.clip(y_pred, eps, 1 - eps)
    return -y_batch * np.log(y_pred) - (1 - y_batch) * np.log(1 - y_pred)

In [ ]:
# Train logistic regression using mini-batch gradient descent
def train_lr(X, y_train, learning_rate, epochs, batch_size, loss_function):
    n = len(X)
    w = np.random.uniform(-0.01, 0.01, size=3)# Randomly initialize (w1, w2, w3)
    rng = np.random.default_rng(2026)

    for epoch in range(epochs):
        order = rng.permutation(n)

        for start in range(0, n, batch_size):
            idx = order[start:start + batch_size]

            X_batch = X[idx]
            y_batch = y_train[idx]

            X_augmented = np.column_stack((X_batch, np.ones(len(X_batch))))# Add constant 1: [x1, x2, 1], homogenious point
            y_pred = sigmoid(X_augmented @ w)# forward pass

            # Hossein's Proposed Error (loss) Function: E = (1 - y_pred) for positive + y_pred for negative
            g = loss_function_hossein(y_batch, y_pred)

            dw = X_augmented.T @ g / len(y_batch) # Gradient for w1, w2, and w3
            w -= learning_rate * dw # Gradient descent

    return w

In [ ]:
from sklearn.model_selection import StratifiedKFold

# Use exactly the same five folds for every combination
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=2026)
folds = list(skf.split(O, y))

# Hyperparameter candidates
learning_rates = [0.01, 0.1, 0.5]
epochs_list = [100, 500, 1000]
batch_sizes = [1, 4, 8]

# Grid search across all hyperparameter combinations
results = []
for lr in learning_rates:
    for epochs in epochs_list:
        for batch_size in batch_sizes:
            precisions, recalls, f_measures = [], [], []

            for fold, (train_idx, test_idx) in enumerate(folds, start=1):
                X_train, X_test = O[train_idx], O[test_idx]
                y_train, y_test = y[train_idx], y[test_idx]

                # Train a fresh model for this fold
                w = train_lr(X_train, y_train, learning_rate=lr, epochs=epochs, batch_size=batch_size, loss_function=loss_function_hossein)

                X_test_augmented = np.column_stack((X_test, np.ones(len(X_test)))) # Add constant 1: [x1, x2, 1], homogenious point

                probabilities = sigmoid(X_test_augmented @ w) # forward pass
                y_pred = (probabilities >= 0.5).astype(float) # predict using a 0.5 probability threshold

                precision, recall, f1 = evaluate(y_test, y_pred)
                precisions.append(precision)
                recalls.append(recall)
                f_measures.append(f1)

            results.append([lr, epochs, batch_size, np.mean(precisions), np.mean(recalls), np.mean(f_measures)])



# Compare mean metrics across five folds
print("\nCROSS-VALIDATION RESULTS")
print("-" * 85)
print(f"{'LR':>6} {'Epochs':>7} {'Batch':>6} {'Precision':>11} {'Recall':>9} {'F-measure':>11}")
print("-" * 85)

for lr, ep, bs, p, r, f in results: print(f"{lr:6.2f} {ep:7d} {bs:6d} {p:11.3f} {r:9.3f} {f:11.3f}")

# Identify the best combination by mean F-measure
best = max(results, key=lambda row: row[5])

print("-" * 85)
print("BEST HYPERPARAMETERS (by mean F-measure)")
print(f"Learning rate = {best[0]}")
print(f"Epochs        = {best[1]}")
print(f"Batch size    = {best[2]}")
print(f"Mean precision = {best[3]:.3f}")
print(f"Mean recall    = {best[4]:.3f}")
print(f"Mean F-measure = {best[5]:.3f}")



CROSS-VALIDATION RESULTS
-------------------------------------------------------------------------------------
    LR  Epochs  Batch   Precision    Recall   F-measure
-------------------------------------------------------------------------------------
  0.01     100      1       0.300     0.600       0.400
  0.01     100      4       0.300     0.600       0.400
  0.01     100      8       0.300     0.600       0.400
  0.01     500      1       0.300     0.600       0.400
  0.01     500      4       0.300     0.600       0.400
  0.01     500      8       0.300     0.600       0.400
  0.01    1000      1       0.300     0.600       0.400
  0.01    1000      4       0.300     0.600       0.400
  0.01    1000      8       0.300     0.600       0.400
  0.10     100      1       0.300     0.600       0.400
  0.10     100      4       0.300     0.600       0.400
  0.10     100      8       0.300     0.600       0.400
  0.10     500      1       0.300     0.600       0.400
  0.10     500    

In [ ]:
loss_functions = [loss_function_hossein, loss_function_standard]
# cross-validate and compare the above results with the standard loss function